# Supplementary Reproducible Analysis Notebook

This notebook reproduces the aggregate findings reported in the paper from sanitized analysis outputs. It is organized to match the Methodology and Results sections:

1. Cohort construction.
2. Validation of the chat-session collection pipeline against the prior conversational-programming package.
3. RQ1 analyses on AI use, chat purposes, and chat-associated commits.
4. RQ2 analyses on repository dynamics before and after AI adoption.
5. RQ3 survey summaries.

The notebook intentionally does **not** display raw developer's demographics, repository owner identities, or raw chat content.

In [1]:
from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if ROOT.name != "vibe-coding":
    # Allows execution from the paper/ directory as well.
    if (ROOT.parent / "outputs").exists():
        ROOT = ROOT.parent
OUT = ROOT / "outputs"
PAPER = ROOT / "paper"

def require(path):
    path = ROOT / path if not isinstance(path, Path) else path
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def read_csv(path, **kwargs):
    return pd.read_csv(require(path), **kwargs)

def pct(x, digits=1):
    return round(float(x) * 100, digits)

def pct_already(x, digits=1):
    return round(float(x), digits)

def round_df(df, digits=3):
    out = df.copy()
    for col in out.select_dtypes(include=[np.number]).columns:
        out[col] = out[col].round(digits)
    return out

SENSITIVE_COL_PAT = re.compile(r"(email|e-mail|address|location|school|institution|raw_text|message_text|content|body)$", re.I)

def assert_no_display_sensitive_columns(df, context="dataframe"):
    bad = [c for c in df.columns if SENSITIVE_COL_PAT.search(str(c))]
    if bad:
        raise AssertionError(f"{context} contains sensitive columns that should not be displayed: {bad}")

def show(df, digits=3):
    assert_no_display_sensitive_columns(df)
    display(round_df(df, digits))

## 1. Cohorts And Paper Output Files

The paper uses the strict toy/homework-filtered dataset:

- Full analysis cohort: 1,240 repositories with complete accessible histories.
- Main pre/post comparison cohort: 608 repositories whose first observed AI chat occurred at or after GitHub publication.
- Older validation cohort: 114 repositories created before 2025.

The following cell verifies these cohort sizes from the final cohort files.

In [2]:
cohort_files = {
    "complete_history_filtered": "outputs/toy_homework_audit/complete_history_repos_excluding_strict_name_toy_homework_1240.csv",
    "main_comparison": "outputs/toy_homework_audit/main_comparison_chat_after_start_excluding_toy_homework_608.csv",
    "older_validation": "outputs/toy_homework_audit/older_pre2025_excluding_toy_homework_114.csv",
}

cohort_summary = []
for label, rel in cohort_files.items():
    df = read_csv(rel)
    cohort_summary.append({"cohort_file": label, "rows": len(df), "columns": len(df.columns)})
cohort_summary = pd.DataFrame(cohort_summary)
show(cohort_summary, 0)

assert int(cohort_summary.loc[cohort_summary.cohort_file == "complete_history_filtered", "rows"].iloc[0]) == 1240
assert int(cohort_summary.loc[cohort_summary.cohort_file == "main_comparison", "rows"].iloc[0]) == 608
assert int(cohort_summary.loc[cohort_summary.cohort_file == "older_validation", "rows"].iloc[0]) == 114

,cohort_file,rows,columns
0,complete_history_filtered,1240,13
1,main_comparison,608,13
2,older_validation,114,13


## 1.1 Analysis Provenance

In [3]:
provenance = pd.DataFrame([
    {
        "paper_section": "Cohort construction / toy-homework filtering",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; py_script_analysis/*toy_homework* audit scripts",
        "aggregate_outputs_used": "outputs/toy_homework_audit/*.csv",
    },
    {
        "paper_section": "Data collection validation for chat sessions",
        "local_script_or_builder": "vibe-coding-scraper-main.zip: scrape.ipynb, parse.py, parse_cli.py",
        "aggregate_outputs_used": "outputs/chat_label_correction_1240/combined_seven_category_summary_1240.csv",
    },
    {
        "paper_section": "RQ1-1 AI-related repository activity",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; py_script_analysis/ai_usage_chat_development_analysis.py",
        "aggregate_outputs_used": "outputs/repo_ai_adoption_filtered_1240/; outputs/ai_related_commit_share_extensions/",
    },
    {
        "paper_section": "RQ1-2 chat purpose and associated commits",
        "local_script_or_builder": "py_script_analysis/collect_missing_paper_label_batches.py; py_script_analysis/chat_purpose_commit_scope_audit.py",
        "aggregate_outputs_used": "outputs/chat_label_correction_1240/; outputs/chat_label_correction_1240/chat_associated_commit_recalc/",
    },
    {
        "paper_section": "RQ2-1 commit activity",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; relative-month commit activity builders",
        "aggregate_outputs_used": "outputs/relative_month_commit_activity_filtered_1240/",
    },
    {
        "paper_section": "RQ2-2 quality and maintenance signals",
        "local_script_or_builder": "build_rq_quality_maintenance_monthly_notebook.py; build_rq_maintenance_followup_interval_notebook.py",
        "aggregate_outputs_used": "outputs/rq_quality_maintenance_filtered_1240/",
    },
    {
        "paper_section": "RQ2-3 issues and pull requests",
        "local_script_or_builder": "py_script_analysis/issue_pr_resolution_activity.py; issue/PR recalculation scripts",
        "aggregate_outputs_used": "outputs/rq_issue_pr_filtered_1240/",
    },
    {
        "paper_section": "RQ2-4 contributors, communication, and review",
        "local_script_or_builder": "build_repo_communication_review_patterns_notebook.py; contributor analysis scripts",
        "aggregate_outputs_used": "outputs/rq_collaboration_review_filtered_1240/",
    },
    {
        "paper_section": "RQ3 survey analysis",
        "local_script_or_builder": "py_script_analysis/analyze_vibe_coding_survey.py",
        "aggregate_outputs_used": "outputs/survey_analysis_all25_no_fdr/; outputs/survey_demographics/",
    },
])
show(provenance, 0)

,paper_section,local_script_or_builder,aggregate_outputs_used
0,Cohort construction / toy-homework filtering,paper/supplementary_data_analysis.py; py_scrip...,outputs/toy_homework_audit/*.csv
1,Data collection validation for chat sessions,"vibe-coding-scraper-main.zip: scrape.ipynb, pa...",outputs/chat_label_correction_1240/combined_se...
2,RQ1-1 AI-related repository activity,paper/supplementary_data_analysis.py; py_scrip...,outputs/repo_ai_adoption_filtered_1240/; outpu...
3,RQ1-2 chat purpose and associated commits,py_script_analysis/collect_missing_paper_label...,outputs/chat_label_correction_1240/; outputs/c...
4,RQ2-1 commit activity,paper/supplementary_data_analysis.py; relative...,outputs/relative_month_commit_activity_filtere...
5,RQ2-2 quality and maintenance signals,build_rq_quality_maintenance_monthly_notebook....,outputs/rq_quality_maintenance_filtered_1240/
6,RQ2-3 issues and pull requests,py_script_analysis/issue_pr_resolution_activit...,outputs/rq_issue_pr_filtered_1240/
7,"RQ2-4 contributors, communication, and review",build_repo_communication_review_patterns_noteb...,outputs/rq_collaboration_review_filtered_1240/
8,RQ3 survey analysis,py_script_analysis/analyze_vibe_coding_survey.py,outputs/survey_analysis_all25_no_fdr/; outputs...


## 1.2 Preprocessing And Derivation Scripts

The preprocessing layer is implemented in the scripts below. These scripts transform collected chat/repository tables into analysis-ready constructs: normalized timestamps, repository keys, file categories, AI-related commit flags, relative months, bot-excluded contributor measures, issue/PR period assignment, and paired pre/post tables.

In [4]:
preprocessing_scripts = pd.DataFrame([
    {
        "script": "paper/supplementary_data_collection.py",
        "role": "Collects GitHub repository histories after chat-session discovery; classifies file paths; hashes email-derived identifiers instead of retaining raw emails.",
        "inputs": "GitHub API responses and chat-session search results",
        "outputs": "clean_data/ and data_complete/repo_activity/ tables",
    },
    {
        "script": "paper/supplementary_data_analysis.py",
        "role": "Main preprocessing and derivation script for analysis-ready tables: UTC timestamps, canonical cohorts, AI-related commit linkage, file flags, relative months, paired pre/post values, ITS panels, and aggregate RQ outputs.",
        "inputs": "clean_data/, data_complete/repo_activity/, outputs/toy_homework_audit/",
        "outputs": "outputs/supplementary_reproducible_analysis/",
    },
    {
        "script": "build_rq_quality_maintenance_monthly_notebook.py",
        "role": "Builds the detailed quality/maintenance notebook and derived monthly quality-signal outputs used in the final paper tables.",
        "inputs": "cleaned commit, issue, file-change, and CI tables",
        "outputs": "outputs/rq_quality_maintenance_filtered_1240/",
    },
    {
        "script": "build_rq_maintenance_followup_interval_notebook.py",
        "role": "Builds same-file rework and bug/fix follow-up interval analyses.",
        "inputs": "cleaned commit and changed-file tables",
        "outputs": "outputs/rq_quality_maintenance_filtered_1240/",
    },
    {
        "script": "build_repo_communication_review_patterns_notebook.py",
        "role": "Builds repository-level communication and review measures.",
        "inputs": "cleaned issue, PR, comment, and review tables",
        "outputs": "outputs/rq_collaboration_review_filtered_1240/",
    },
])

for script in preprocessing_scripts["script"]:
    assert require(script).exists()
show(preprocessing_scripts, 0)

,script,role,inputs,outputs
0,paper/supplementary_data_collection.py,Collects GitHub repository histories after cha...,GitHub API responses and chat-session search r...,clean_data/ and data_complete/repo_activity/ t...
1,paper/supplementary_data_analysis.py,Main preprocessing and derivation script for a...,"clean_data/, data_complete/repo_activity/, out...",outputs/supplementary_reproducible_analysis/
2,build_rq_quality_maintenance_monthly_notebook.py,Builds the detailed quality/maintenance notebo...,"cleaned commit, issue, file-change, and CI tables",outputs/rq_quality_maintenance_filtered_1240/
3,build_rq_maintenance_followup_interval_noteboo...,Builds same-file rework and bug/fix follow-up ...,cleaned commit and changed-file tables,outputs/rq_quality_maintenance_filtered_1240/
4,build_repo_communication_review_patterns_noteb...,Builds repository-level communication and revi...,"cleaned issue, PR, comment, and review tables",outputs/rq_collaboration_review_filtered_1240/


## 2. Chat-Session Collection Validation

The prior paper reports 11,579 AI-assisted IDE chat sessions across 1,300 repositories and 74,998 developer messages. Its public replication package documents the expected data layout, including GitHub search results, raw Markdown chat histories, parsed chats, per-file commit histories, and repository commit histories. Our chat-session scraper follows the same collection structure.

In [5]:
scraper_zip = require("vibe-coding-scraper-main.zip")
with zipfile.ZipFile(scraper_zip) as zf:
    zip_names = zf.namelist()
    readme_name = next(n for n in zip_names if n.endswith("ReadMe.md"))
    parse_name = next(n for n in zip_names if n.endswith("parse.py"))
    parse_cli_name = next(n for n in zip_names if n.endswith("parse_cli.py"))
    scraper_readme = zf.read(readme_name).decode("utf-8", errors="ignore")
    parse_text = zf.read(parse_name).decode("utf-8", errors="ignore")
    parse_cli_text = zf.read(parse_cli_name).decode("utf-8", errors="ignore")

expected_collection_artifacts = [
    "searches/",
    "searches.json",
    "repositories.json",
    "markdowns/",
    "markdowns_cli/",
    "parsed_chats/",
    "parsed_chats_simple/",
    "parsed_chats_simple_cli/",
    "commits_path/",
    "commits/",
    "commits_history/",
    "languages.json",
]

validation_rows = []
for artifact in expected_collection_artifacts:
    validation_rows.append({
        "expected_artifact": artifact,
        "declared_in_local_scraper_readme": artifact in scraper_readme,
    })

parser_checks = pd.DataFrame([
    {"check": "SpecStory path/search marker", "present": ".specstory" in scraper_readme or ".specstory" in parse_text},
    {"check": "standard role parser: User", "present": "User" in parse_text},
    {"check": "standard role parser: Assistant/Agent", "present": ("Assistant" in parse_text) or ("Agent" in parse_text)},
    {"check": "CLI parser included", "present": len(parse_cli_text) > 1000},
])

show(pd.DataFrame(validation_rows), 0)
show(parser_checks, 0)
assert all(r["declared_in_local_scraper_readme"] for r in validation_rows)
assert parser_checks["present"].all()

,expected_artifact,declared_in_local_scraper_readme
0,searches/,True
1,searches.json,True
2,repositories.json,True
3,markdowns/,True
4,markdowns_cli/,True
5,parsed_chats/,True
6,parsed_chats_simple/,True
7,parsed_chats_simple_cli/,True
8,commits_path/,True
9,commits/,True


,check,present
0,SpecStory path/search marker,True
1,standard role parser: User,True
2,standard role parser: Assistant/Agent,True
3,CLI parser included,True


The next cell verifies the aggregate chat-session counts used in the paper after applying the strict repository filter.

In [6]:
chat_summary = read_csv("outputs/chat_label_correction_1240/combined_seven_category_summary_1240.csv")
show(chat_summary, 3)

assert int(chat_summary.loc[0, "mapped_chat_sessions"]) == 12108
assert int(chat_summary.loc[0, "mapped_repositories"]) == 1240

,mapped_chat_sessions,mapped_repositories,mapped_sessions_mean_per_repo,mapped_sessions_median_per_repo,combined_labeled_sessions_including_others,others_sessions,seven_category_labeled_sessions,seven_category_labeled_repositories,mean_distinct_seven_purposes_per_labeled_repo,median_distinct_seven_purposes_per_labeled_repo
0,12108,1240,9.765,2.0,9240,65,9175,1128,2.135,2.0


## 3. RQ1-1: AI-Related Repository Activity

This section reproduces the aggregate post-adoption AI-related commit measures and file-change composition reported in the paper. AI-related commits are identified from artifact-containing commits; AI-related development activity focuses on mixed commits that also changed development files.

In [7]:
post_ai = read_csv("outputs/repo_ai_adoption_filtered_1240/repo_post_adoption_ai_commit_summary.csv")
rq1_summary = pd.DataFrame([{
    "repositories_with_post_adoption_commits": len(post_ai),
    "mean_post_adoption_commits": post_ai["post_commits"].mean(),
    "median_post_adoption_commits": post_ai["post_commits"].median(),
    "mean_ai_related_commits": post_ai["ai_related_commits"].mean(),
    "median_ai_related_commits": post_ai["ai_related_commits"].median(),
    "mean_ai_related_commit_share_pct": post_ai["ai_commit_share"].mean() * 100,
    "median_ai_related_commit_share_pct": post_ai["ai_commit_share"].median() * 100,
}])
show(rq1_summary, 2)

,repositories_with_post_adoption_commits,mean_post_adoption_commits,median_post_adoption_commits,mean_ai_related_commits,median_ai_related_commits,mean_ai_related_commit_share_pct,median_ai_related_commit_share_pct
0,1237,63.9,11.0,8.37,3.0,44.33,34.29


In [8]:
file_comp = read_csv("outputs/repo_ai_adoption_filtered_1240/ai_related_commit_file_type_distribution.csv")
size = read_csv("outputs/repo_ai_adoption_filtered_1240/post_adoption_observable_ai_commit_size.csv")

display_cols = [c for c in file_comp.columns if not SENSITIVE_COL_PAT.search(str(c))]
show(file_comp[display_cols], 2)

size_summary = pd.DataFrame([{
    "ai_related_commits": len(size),
    "mean_files_changed": size["files_changed"].mean() if "files_changed" in size else np.nan,
    "median_files_changed": size["files_changed"].median() if "files_changed" in size else np.nan,
    "mean_total_churn": size["total_churn"].mean() if "total_churn" in size else np.nan,
    "median_total_churn": size["total_churn"].median() if "total_churn" in size else np.nan,
}])
show(size_summary, 2)

,file_type,file_changes_x,share_all_ai_file_changes_pct,file_changes_y,share_development_file_changes_pct
0,source_code,99891,37.51,99891.0,43.56
1,other,47914,17.99,47914.0,20.89
2,documentation,34495,12.95,34495.0,15.04
3,generated_binary,27064,10.16,27064.0,11.80
4,configuration,7978,3.00,7978.0,3.48
5,test,7370,2.77,7370.0,3.21
6,dependency,4598,1.73,4598.0,2.01
7,ai_artifact,36982,13.89,0.0,0.00


,ai_related_commits,mean_files_changed,median_files_changed,mean_total_churn,median_total_churn
0,10355,25.72,7.0,19947.03,1789.0


In [9]:
trend = read_csv("outputs/ai_related_commit_share_extensions/ai_commit_share_by_relative_month_0_12.csv")
size_trend = read_csv("outputs/ai_related_commit_share_extensions/ai_related_commit_size_by_relative_month_0_12.csv")
show(trend.head(13), 3)
show(size_trend.head(13), 3)

,relative_month,repos,total_commits,ai_related_commits,mean_repo_month_ai_share,median_repo_month_ai_share,pooled_ai_share
0,0,995,20204,5072,0.506,0.500,0.251
1,1,398,11323,1992,0.328,0.145,0.176
2,2,240,6631,838,0.284,0.026,0.126
3,3,177,7782,492,0.189,0.000,0.063
4,4,143,3871,358,0.180,0.000,0.092
5,5,104,1927,246,0.181,0.000,0.128
6,6,91,1540,214,0.161,0.000,0.139
7,7,48,803,99,0.213,0.000,0.123
8,8,52,900,21,0.145,0.000,0.023
9,9,35,483,14,0.118,0.000,0.029


,relative_month,commits,repos,total_churn_mean,total_churn_median,total_churn_p75,total_churn_p90,source_churn_mean,source_churn_median,source_churn_p75,source_churn_p90,files_changed_mean,files_changed_median,files_changed_p75,files_changed_p90,source_files_changed_mean,source_files_changed_median,source_files_changed_p75,source_files_changed_p90
0,0,5692,996,15473.265,1638.5,6244.25,19374.2,2211.012,113.0,681.25,2752.8,23.669,7.0,18.00,52.9,9.129,2.0,7.00,20.0
1,1,2188,296,24485.203,2023.5,9751.00,41487.4,1979.533,113.0,824.50,3202.0,25.899,7.0,19.00,61.0,9.802,2.0,8.00,22.0
2,2,926,139,23536.148,1556.0,6022.75,25330.0,2900.962,53.5,508.00,2583.5,28.011,6.0,15.00,67.0,10.911,1.0,5.00,22.0
3,3,528,83,23761.947,1992.5,8566.50,34218.4,2148.513,41.0,488.75,3420.5,33.562,7.5,20.00,87.2,11.201,1.0,7.00,26.0
4,4,300,67,16635.247,1628.5,8260.00,40113.9,984.167,40.0,604.00,1980.9,26.463,7.0,18.00,56.1,9.143,1.0,6.00,18.0
5,5,250,48,22285.380,1984.5,7271.00,25701.2,1296.928,24.0,429.50,2126.9,23.420,8.0,18.75,53.1,8.296,1.0,5.00,15.0
6,6,242,40,19187.855,4014.5,15528.00,35201.7,2206.707,21.0,243.00,1100.1,22.913,5.0,11.00,40.0,7.744,1.0,3.00,11.9
7,7,95,21,32410.695,1158.0,20053.50,58297.6,1805.474,25.0,342.50,3019.2,40.316,7.0,33.50,116.8,17.337,2.0,6.50,63.2
8,8,39,25,57155.897,2249.0,19260.50,71559.8,1762.308,0.0,74.00,6738.4,72.513,19.0,93.50,300.0,19.821,1.0,12.00,69.8
9,9,30,12,11095.167,2738.5,10363.00,36477.6,3533.633,280.5,2541.00,17731.5,62.100,19.5,52.75,300.0,20.667,7.0,19.00,61.4


## 4. RQ1-2: Chat Purpose And Chat-Associated Development

Chat-purpose labels are multi-label. Therefore, category percentages are prevalence rates rather than mutually exclusive shares, and can sum above 100%.

In [10]:
purpose = read_csv("outputs/chat_label_correction_1240/corrected_multilabel_chat_purpose_distribution_1240.csv")
dominant = read_csv("outputs/chat_label_correction_1240/combined_repository_dominant_category_distribution_1240.csv")
purpose_display = purpose[[
    "Purpose", "Msg_prevalence_all_labeled_%", "Session_prevalence_%", "Sessions_with_purpose"
]].rename(columns={
    "Msg_prevalence_all_labeled_%": "message_prevalence_pct",
    "Session_prevalence_%": "session_prevalence_pct",
})
show(purpose_display, 1)
show(dominant, 1)

,Purpose,message_prevalence_pct,session_prevalence_pct,Sessions_with_purpose
0,Code Writing,34.7,69.6,6404
1,Failure Reporting,24.2,48.1,4421
2,Delegation,16.0,42.6,3915
3,Inquiry,19.3,42.8,3933
4,Context Spec.,14.1,43.0,3951
5,Workflow Control,11.5,32.4,2976
6,Validation,3.9,17.4,1601


,category,repositories,percent
0,Code Authoring,605,53.5
1,Delegation,175,15.5
2,Failure Reporting,142,12.6
3,Inquiry,100,8.8
4,Context Specification,59,5.2
5,Workflow Control,37,3.3
6,Others,8,0.7
7,Validation,5,0.4


In [11]:
assoc_overall = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/chat_associated_commit_overall_summary.csv")
assoc_by_purpose = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_by_purpose_summary.csv")
assoc_scope = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_scope_by_chat_type.csv")
assoc_tests = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_scope_tests.csv")

show(assoc_overall, 3)
show(assoc_by_purpose, 2)
show(assoc_scope, 2)
show(assoc_tests, 3)

,labeled_sessions_filtered,resolved_chat_associated_commit_sessions,resolved_pct,sessions_with_development_files,development_pct_of_labeled,development_pct_of_resolved,file_type_cramers_v,file_type_chi2_p,file_type_repo_blocked_perm_p
0,9240,9139,98.907,8877,96.071,97.133,0.126,0.0,1.0


,category,label_sessions,resolved_pct,development_pct_of_label_sessions,development_sessions,source_only_pct,docs_only_pct,config_only_pct,source_test_docs_config_dependency_pct
0,Code Authoring,6404,98.78,95.86,6139,29.19,15.96,4.64,7.20
1,Failure Reporting,4421,99.05,96.40,4262,29.77,15.20,7.95,6.85
2,Delegation,3915,98.57,94.25,3690,26.86,20.16,3.60,8.92
3,Inquiry,3933,99.14,94.79,3728,30.39,19.26,4.24,8.42
4,Context Specification,3951,98.79,94.13,3719,30.01,16.40,3.85,7.21
5,Workflow Control,2976,98.96,93.85,2793,29.90,17.22,4.12,10.31
6,Validation,1601,99.13,93.32,1494,27.51,14.79,6.56,10.44


,category,n_sessions,n_repos,total_churn_mean,total_churn_median,files_changed_mean,files_changed_median,file_type_categories_mean,file_type_categories_median,source_churn_mean,source_churn_median,source_files_changed_mean,source_files_changed_median
0,Code Authoring,6139,931,72543.41,989.0,52.04,7.0,2.56,2.0,3694.63,67.0,16.81,2.0
1,Failure Reporting,4262,809,21783.43,470.0,33.84,5.0,2.43,2.0,3304.67,41.0,15.51,1.0
2,Delegation,3690,830,19959.87,771.0,32.83,6.0,2.50,2.0,3205.64,43.0,13.07,1.0
3,Inquiry,3728,814,24581.62,485.0,32.59,5.0,2.43,2.0,2933.98,42.0,12.98,1.0
4,Context Specification,3719,791,56628.98,921.0,46.54,7.0,2.56,2.0,3807.41,88.0,16.37,2.0
5,Workflow Control,2793,645,20416.50,748.0,33.09,6.0,2.59,2.0,3335.90,69.0,14.84,2.0
6,Validation,1494,467,26956.55,813.0,36.41,7.0,2.67,2.0,2955.26,83.0,13.89,2.0


,metric,blocked_permutation_h,p_value,q_value
0,total_churn,133.776,0.96,1.0
1,files_changed,146.172,0.96,1.0
2,file_type_categories,58.459,0.96,1.0
3,source_churn,117.104,1.00,1.0
4,source_files_changed,131.795,1.00,1.0


## 5. RQ2-1: Commit Activity Before And After AI Adoption

The main RQ2 comparison uses the 608 post-publication repositories. Months are re-indexed relative to the AI-adoption month; the adoption month itself is excluded from pre/post comparisons.

In [ ]:
commit_tests = read_csv("outputs/relative_month_commit_activity_filtered_1240/relative_month_pre_post_activity_tests_filtered.csv")
commit_its = read_csv("outputs/relative_month_commit_activity_filtered_1240/relative_month_its_pre_post_slope_models_filtered.csv")

show(commit_tests, 3)
show(commit_its[commit_its["cohort"].isin(["main_608", "older_114"])], 4)

## 6. RQ2-2: Observable Code-Quality And Maintenance Signals

The following tables reproduce the defect, testing/CI, rework, bug/fix follow-up, revert, and follow-up interval summaries. The first group reports repository-period aggregate values; the ITS model outputs are kept separately for temporal trend checks.

In [ ]:
defect_raw = read_csv("outputs/rq_quality_maintenance_filtered_1240/defect_recalculation/defect_raw_repo_period_tests.csv")
bug_issue_share = read_csv("outputs/rq_quality_maintenance_filtered_1240/bug_issue_share_recalc/bug_issue_share_pre_post_tests.csv")
testing_ci = read_csv("outputs/rq_quality_maintenance_filtered_1240/general_prepost_recalc/testing_ci_general_repo_period_tests_608.csv")
rework = read_csv("outputs/rq_quality_maintenance_filtered_1240/general_prepost_recalc/rework_bugfix_revert_general_repo_period_tests_608.csv")
intervals = read_csv("outputs/rq_quality_maintenance_filtered_1240/followup_interval_tests_filtered_608_114.csv")
qm_its = read_csv("outputs/rq_quality_maintenance_filtered_1240/quality_maintenance_its_recalc/quality_maintenance_relative_month_its_models_filtered_608_114.csv")

show(defect_raw, 4)
show(bug_issue_share, 4)
show(testing_ci, 4)
show(rework, 4)
show(intervals, 4)
show(qm_its.head(20), 4)

## 7. RQ2-3: Issues And Pull Requests

Issue and PR volume is summarized over each repository's full pre/post period. Share measures distinguish issue vs PR composition among opened items and closure/merge rates among eligible items. Efficiency measures use repository-level issue resolution and PR merge times.

In [ ]:
issue_pr_counts = read_csv("outputs/rq_issue_pr_filtered_1240/general_prepost_recalc/issue_pr_general_repo_period_tests_608.csv")
issue_pr_shares = read_csv("outputs/rq_issue_pr_filtered_1240/whole_period_shares/issue_pr_whole_period_share_tests_608_114.csv")
issue_pr_eff = read_csv("outputs/rq_issue_pr_filtered_1240/repo_level_issue_pr_efficiency_tests_filtered_608_114.csv")
issue_pr_its = read_csv("outputs/rq_issue_pr_filtered_1240/its_recalc/relative_month_issue_pr_its_models_filtered_608_114.csv")

show(issue_pr_counts, 4)
show(issue_pr_shares, 4)
show(issue_pr_eff, 4)
show(issue_pr_its.head(24), 4)

## 8. RQ2-4: Contributor Participation, Communication, And Review

Contributor participation is measured at the repository level over full pre/post periods. Communication and review measures are also repository-level paired comparisons, with denominator-zero ratios treated as missing.

In [ ]:
contributors = read_csv("outputs/rq_collaboration_review_filtered_1240/full_window_repository_contributor_tests_filtered_608_114.csv")
active_share = read_csv("outputs/rq_collaboration_review_filtered_1240/active_contributor_share/active_contributor_share_tests.csv")
comm_review = read_csv("outputs/rq_collaboration_review_filtered_1240/communication_review_paired_tests_filtered_608_114.csv")

show(contributors, 4)
show(active_share, 4)
show(comm_review, 4)

## 9. RQ3: Survey Findings

This section uses only aggregate survey outputs. Raw survey records, names, emails, free-text identifiers, and locations are not loaded or displayed.

In [ ]:
constructs = read_csv("outputs/survey_analysis_all25_no_fdr/construct_summary_all25_no_fdr.csv")
self_vs_others = read_csv("outputs/survey_analysis_all25_no_fdr/self_vs_others_all25_no_fdr.csv")
appropriate = read_csv("outputs/survey_analysis_all25_no_fdr/Q17_multiselect_exact_counts_all25.csv")
avoid = read_csv("outputs/survey_analysis_all25_no_fdr/Q18_multiselect_exact_counts_all25.csv")
demo = read_csv("outputs/survey_demographics/all25_demographic_distribution_grouped.csv")

show(constructs, 4)
show(self_vs_others, 4)
show(appropriate, 2)
show(avoid, 2)
show(demo, 2)